# AGFB Generator Visual Check

This notebook renders every generator currently exported by the package, runs lightweight shape and gradient checks, and displays intensity and gradient views. It uses a small PNG renderer instead of plotting libraries.

In [ ]:
from __future__ import annotations

import base64
import html
import math
import struct
import zlib

import numpy as np
import torch

from agfb_generators import (
    CompositeRect,
    Frame,
    composite,
    curved_arc,
    gaussian_blob,
    gaussian_ridge,
    hard_step,
    polynomial,
    sinusoid,
    smoothed_bar,
    smoothed_step,
)

try:
    from IPython.display import HTML, display
except ImportError:

    class HTML:  # type: ignore[no-redef]
        def __init__(self, data: str) -> None:
            self.data = data

    def display(obj: object) -> None:  # type: ignore[no-redef]
        print(getattr(obj, "data", obj))

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
HEIGHT = 160
WIDTH = 160

BRAND = {
    "garnet": "#73000A",
    "black": "#000000",
    "white": "#FFFFFF",
    "gray90": "#363636",
    "gray70": "#5C5C5C",
    "gray50": "#A2A2A2",
    "gray30": "#C7C7C7",
    "gray10": "#ECECEC",
    "rose": "#CC2E40",
    "atlantic": "#466A9F",
    "congaree": "#1F414D",
    "grass": "#CED318",
    "honeycomb": "#A49137",
}

PALETTE_INTENSITY = [
    (0.0, BRAND["black"]),
    (0.55, BRAND["garnet"]),
    (1.0, BRAND["white"]),
]
PALETTE_MAGNITUDE = [
    (0.0, BRAND["black"]),
    (0.72, BRAND["honeycomb"]),
    (1.0, BRAND["white"]),
]
PALETTE_SIGNED = [
    (0.0, BRAND["rose"]),
    (0.5, BRAND["white"]),
    (1.0, BRAND["atlantic"]),
]
PALETTE_MASK = [
    (0.0, BRAND["black"]),
    (1.0, BRAND["grass"]),
]

In [ ]:
def _hex_to_rgb(color: str) -> np.ndarray:
    color = color.lstrip("#")
    return np.array([int(color[i : i + 2], 16) for i in (0, 2, 4)], dtype=np.float32)


def _normalize(values: np.ndarray, *, symmetric: bool = False) -> np.ndarray:
    data = np.asarray(values, dtype=np.float32)
    mask = np.isfinite(data)
    if not mask.any():
        return np.zeros_like(data, dtype=np.float32)
    if symmetric:
        limit = float(np.max(np.abs(data[mask])))
        if limit <= 1e-12:
            return np.full_like(data, 0.5, dtype=np.float32)
        return np.clip(0.5 + 0.5 * data / limit, 0.0, 1.0)
    lo = float(np.min(data[mask]))
    hi = float(np.max(data[mask]))
    if hi - lo <= 1e-12:
        return np.zeros_like(data, dtype=np.float32)
    return np.clip((data - lo) / (hi - lo), 0.0, 1.0)


def _apply_palette(values: np.ndarray, palette: list[tuple[float, str]]) -> np.ndarray:
    positions = np.array([p for p, _ in palette], dtype=np.float32)
    colors = np.stack([_hex_to_rgb(c) for _, c in palette], axis=0)
    flat = np.asarray(values, dtype=np.float32).reshape(-1)
    channels = [np.interp(flat, positions, colors[:, k]) for k in range(3)]
    rgb = np.stack(channels, axis=1).reshape(*values.shape, 3)
    return np.clip(rgb, 0, 255).astype(np.uint8)


def _png_chunk(tag: bytes, data: bytes) -> bytes:
    payload = tag + data
    checksum = zlib.crc32(payload) & 0xFFFFFFFF
    return struct.pack(">I", len(data)) + payload + struct.pack(">I", checksum)


def _png_rgb(rgb: np.ndarray) -> bytes:
    rgb = np.asarray(rgb, dtype=np.uint8)
    height, width, channels = rgb.shape
    if channels != 3:
        raise ValueError(f"expected RGB image, got shape {rgb.shape}")
    header = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    raw = b"".join(b"\x00" + rgb[row].tobytes() for row in range(height))
    return (
        b"\x89PNG\r\n\x1a\n"
        + _png_chunk(b"IHDR", header)
        + _png_chunk(b"IDAT", zlib.compress(raw))
        + _png_chunk(b"IEND", b"")
    )


def _image_uri(
    values: np.ndarray,
    palette: list[tuple[float, str]],
    *,
    symmetric: bool = False,
) -> str:
    normalized = _normalize(values, symmetric=symmetric)
    rgb = _apply_palette(normalized, palette)
    encoded = base64.b64encode(_png_rgb(rgb)).decode("ascii")
    return f"data:image/png;base64,{encoded}"


def _figure(
    values: np.ndarray,
    title: str,
    palette: list[tuple[float, str]],
    *,
    symmetric: bool = False,
) -> str:
    uri = _image_uri(values, palette, symmetric=symmetric)
    caption = html.escape(title)
    return (
        "<figure class='agfb-figure'>"
        f"<figcaption>{caption}</figcaption>"
        f"<img src='{uri}' alt='{caption}' />"
        "</figure>"
    )


display(
    HTML(
        f"""
        <style>
        .agfb-block {{ border-top: 2px solid {BRAND["garnet"]}; padding: 12px 0 18px; }}
        .agfb-block h3 {{ margin: 0 0 8px; font-size: 16px; color: {BRAND["black"]}; }}
        .agfb-grid {{
            display: grid;
            grid-template-columns: repeat(4, minmax(120px, 1fr));
            gap: 10px;
        }}
        .agfb-figure {{
            margin: 0;
            border: 1px solid {BRAND["gray30"]};
            background: {BRAND["white"]};
        }}
        .agfb-figure figcaption {{
            padding: 5px 6px;
            font: 12px sans-serif;
            background: {BRAND["gray10"]};
            color: {BRAND["black"]};
        }}
        .agfb-figure img {{
            display: block;
            width: 100%;
            image-rendering: pixelated;
        }}
        .agfb-table {{
            border-collapse: collapse;
            width: 100%;
            font: 13px sans-serif;
        }}
        .agfb-table th {{
            background: {BRAND["black"]};
            color: {BRAND["white"]};
            text-align: left;
        }}
        .agfb-table th, .agfb-table td {{
            border: 1px solid {BRAND["gray30"]};
            padding: 6px 8px;
        }}
        </style>
        """
    )
)

In [ ]:
def _fd4(image: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    gx = torch.zeros_like(image)
    gy = torch.zeros_like(image)
    gx[:, 2:-2] = (-image[:, 4:] + 8 * image[:, 3:-1] - 8 * image[:, 1:-3] + image[:, :-4]) / 12.0
    gy[2:-2, :] = (-image[4:, :] + 8 * image[3:-1, :] - 8 * image[1:-3, :] + image[:-4, :]) / 12.0
    return gx, gy


def _check_frame(name: str, frame: Frame, *, rel_tol: float) -> dict[str, object]:
    assert frame.I.ndim == 3, f"{name}: expected I shape (B, H, W), got {tuple(frame.I.shape)}"
    assert frame.g.ndim == 4, f"{name}: expected g shape (B, 2, H, W), got {tuple(frame.g.shape)}"
    assert frame.g.shape[1] == 2, f"{name}: gradient channel count must be 2"
    assert frame.I.shape[0] == frame.g.shape[0], f"{name}: batch mismatch"
    assert frame.I.shape[1:] == frame.g.shape[2:], f"{name}: spatial shape mismatch"
    assert torch.isfinite(frame.I).all(), f"{name}: non-finite intensity"
    assert torch.isfinite(frame.g).all(), f"{name}: non-finite gradient"

    image = frame.I[0]
    fd_gx, fd_gy = _fd4(image)
    inner = torch.zeros_like(image, dtype=torch.bool)
    inner[3:-3, 3:-3] = True
    mag = torch.sqrt(frame.gx[0] ** 2 + frame.gy[0] ** 2)
    signal = (mag > 1e-2 * float(mag.max())) & inner
    signal_count = int(signal.sum())
    assert signal_count > 50, f"{name}: signal mask too small ({signal_count})"

    diff_x = (fd_gx - frame.gx[0])[signal]
    diff_y = (fd_gy - frame.gy[0])[signal]
    numerator = torch.mean(diff_x * diff_x + diff_y * diff_y)
    denominator = torch.mean(frame.gx[0][signal] ** 2 + frame.gy[0][signal] ** 2)
    nrmse = float(torch.sqrt(numerator / denominator))
    assert nrmse < rel_tol, f"{name}: NRMSE={nrmse:.3e} >= {rel_tol:.3e}"

    return {
        "name": name,
        "batch": frame.batch_size,
        "shape": f"{frame.height}x{frame.width}",
        "max_grad": float(mag.max()),
        "signal_px": signal_count,
        "nrmse": nrmse,
        "tol": rel_tol,
        "status": "pass",
    }


def _render_table(rows: list[dict[str, object]]) -> None:
    headers = ["name", "batch", "shape", "max_grad", "signal_px", "nrmse", "tol", "status"]
    head = "".join(f"<th>{html.escape(h)}</th>" for h in headers)
    body_rows = []
    for row in rows:
        cells = []
        for header in headers:
            value = row.get(header, "")
            if isinstance(value, float):
                value = f"{value:.3e}"
            cells.append(f"<td>{html.escape(str(value))}</td>")
        body_rows.append("<tr>" + "".join(cells) + "</tr>")
    display(
        HTML(
            "<table class='agfb-table'><thead><tr>"
            + head
            + "</tr></thead><tbody>"
            + "".join(body_rows)
            + "</tbody></table>"
        )
    )

In [ ]:
coeffs = torch.zeros(1, 4, 4, device=DEVICE, dtype=DTYPE)
coeffs[0, 1, 0] = 0.3
coeffs[0, 0, 1] = -0.2
coeffs[0, 2, 1] = 0.05
coeffs[0, 1, 2] = -0.04

CASES: list[tuple[str, Frame, float]] = [
    (
        "smoothed_step",
        smoothed_step(
            HEIGHT, WIDTH, theta_rad=math.radians(30.0), sigma_e=4.0, device=DEVICE, dtype=DTYPE
        ),
        1e-3,
    ),
    (
        "hard_step",
        hard_step(HEIGHT, WIDTH, theta_rad=math.radians(15.0), device=DEVICE, dtype=DTYPE),
        3e-1,
    ),
    (
        "curved_arc",
        curved_arc(HEIGHT, WIDTH, r0=42.0, sigma_e=4.0, device=DEVICE, dtype=DTYPE),
        1e-3,
    ),
    (
        "sinusoid",
        sinusoid(
            HEIGHT, WIDTH, freq=0.05, theta_rad=math.radians(30.0), device=DEVICE, dtype=DTYPE
        ),
        1e-2,
    ),
    (
        "gaussian_blob",
        gaussian_blob(HEIGHT, WIDTH, sigma=8.0, x0=-10.0, y0=8.0, device=DEVICE, dtype=DTYPE),
        1e-3,
    ),
    (
        "gaussian_ridge",
        gaussian_ridge(
            HEIGHT, WIDTH, sigma=4.0, theta_rad=math.radians(20.0), device=DEVICE, dtype=DTYPE
        ),
        1e-3,
    ),
    (
        "smoothed_bar",
        smoothed_bar(
            HEIGHT,
            WIDTH,
            width_px=32.0,
            theta_rad=math.radians(15.0),
            sigma_e=4.0,
            device=DEVICE,
            dtype=DTYPE,
        ),
        1e-3,
    ),
    (
        "polynomial",
        polynomial(HEIGHT, WIDTH, coeffs=coeffs, scale=64.0, device=DEVICE, dtype=DTYPE),
        1e-3,
    ),
]

rects = [
    CompositeRect(0, HEIGHT // 2, 0, WIDTH // 2, CASES[0][1]),
    CompositeRect(0, HEIGHT // 2, WIDTH // 2, WIDTH, CASES[4][1]),
    CompositeRect(HEIGHT // 2, HEIGHT, 0, WIDTH // 2, CASES[5][1]),
    CompositeRect(HEIGHT // 2, HEIGHT, WIDTH // 2, WIDTH, CASES[3][1]),
]
COMPOSITE_FRAME, COMPOSITE_JUNCTION = composite(
    HEIGHT,
    WIDTH,
    rects,
    junction_radius=3,
    device=DEVICE,
    dtype=DTYPE,
)

In [ ]:
rows = [_check_frame(name, frame, rel_tol=tol) for name, frame, tol in CASES]

assert COMPOSITE_FRAME.I.shape == (1, HEIGHT, WIDTH)
assert COMPOSITE_FRAME.g.shape == (1, 2, HEIGHT, WIDTH)
assert COMPOSITE_JUNCTION.shape == (HEIGHT, WIDTH)
assert torch.isfinite(COMPOSITE_FRAME.I).all()
assert torch.isfinite(COMPOSITE_FRAME.g).all()
rows.append(
    {
        "name": "composite",
        "batch": COMPOSITE_FRAME.batch_size,
        "shape": f"{COMPOSITE_FRAME.height}x{COMPOSITE_FRAME.width}",
        "max_grad": float(
            torch.sqrt(COMPOSITE_FRAME.gx[0] ** 2 + COMPOSITE_FRAME.gy[0] ** 2).max()
        ),
        "signal_px": int(COMPOSITE_JUNCTION.sum()),
        "nrmse": float("nan"),
        "tol": float("nan"),
        "status": "pass",
    }
)

batch = smoothed_step(
    HEIGHT,
    WIDTH,
    theta_rad=torch.tensor([0.0, math.radians(30.0), math.radians(60.0)], device=DEVICE),
    sigma_e=torch.tensor([2.0, 4.0, 6.0], device=DEVICE),
    device=DEVICE,
    dtype=DTYPE,
)
assert batch.batch_size == 3
assert batch.I.shape == (3, HEIGHT, WIDTH)
assert batch.g.shape == (3, 2, HEIGHT, WIDTH)

_render_table(rows)

In [ ]:
def show_frame(name: str, frame: Frame, *, index: int = 0) -> None:
    image = frame.I[index].detach().cpu().numpy()
    gx = frame.gx[index].detach().cpu().numpy()
    gy = frame.gy[index].detach().cpu().numpy()
    grad_mag = np.sqrt(gx * gx + gy * gy)
    block = (
        "<section class='agfb-block'>"
        f"<h3>{html.escape(name)}</h3>"
        "<div class='agfb-grid'>"
        + _figure(image, "intensity", PALETTE_INTENSITY)
        + _figure(grad_mag, "gradient magnitude", PALETTE_MAGNITUDE)
        + _figure(gx, "g_x", PALETTE_SIGNED, symmetric=True)
        + _figure(gy, "g_y", PALETTE_SIGNED, symmetric=True)
        + "</div></section>"
    )
    display(HTML(block))


for case_name, case_frame, _ in CASES:
    show_frame(case_name, case_frame)

show_frame("composite", COMPOSITE_FRAME)

In [ ]:
mask = COMPOSITE_JUNCTION.detach().cpu().numpy().astype(np.float32)
display(
    HTML(
        "<section class='agfb-block'><h3>composite junction mask</h3>"
        "<div class='agfb-grid'>"
        + _figure(mask, "junction mask", PALETTE_MASK)
        + "</div></section>"
    )
)

print(f"Rendered {len(CASES)} generator examples plus one composite on {DEVICE}.")